# 🎙️ LifeMate v3.0 — Adaptive Voice-to-Action Agent Pipeline
### Zalo AI Challenge 2026

This notebook implements the complete **v3.0 Adaptive Agent Pipeline**:
1. **Audio Tai Nghe**: `G-Group Gipformer 1.5` (68M Zipformer-RNNT on CPU via Sherpa-ONNX) — 0% GPU VRAM used, ultra-lightweight and robust on Vietnamese conversational audio.
2. **Skill Tri Thức**: `SkillGraph` (Adaptive Knowledge Graph engine) — Models input-output tool dependencies, eliminates hardcoding, and dynamically prunes irrelevant distractor domains.
3. **Agent Não Bộ**: Adaptive SFT Meta-Reasoning (`<thought>` tags, tool order shuffling, distractor injection) on `Qwen/Qwen2.5-3B-Instruct` (strictly < 5B parameters constraint).
4. **Loss Masking**: Completion-Only Loss Collator ensuring 100% gradient updates on thoughts and tool calls.
5. **Auto-Merge**: Merges LoRA weights into a standalone offline model for zero-latency inference.
6. **Official Validation**: 100% compliant with competition rules (deterministic greedy decoding `seed=42`, `zaic_tools` validation, exactly 634 lines).

In [ ]:
# 1. GPU Check
!nvidia-smi

In [ ]:
# 2. Install Core Dependencies
!pip install -q -U "transformers>=4.48.0" soundfile accelerate peft trl datasets sherpa-onnx huggingface_hub
!pip uninstall -y torchao

In [ ]:
# 3. Clone Repository
import os
REPO_URL = "https://github.com/Quandinh030106/LifeMate-Voice2Action-Agent.git"
if not os.path.exists("/kaggle/working/LifeMate-Voice2Action-Agent"):
    !git clone $REPO_URL /kaggle/working/LifeMate-Voice2Action-Agent
else:
    !cd /kaggle/working/LifeMate-Voice2Action-Agent && git fetch origin && git reset --hard origin/main

%cd /kaggle/working/LifeMate-Voice2Action-Agent

In [ ]:
# 4. Locate Dataset Directory
import glob, os
datasets = glob.glob("/kaggle/input/**/public_test.jsonl", recursive=True)
if datasets:
    DATA_DIR = os.path.dirname(os.path.dirname(datasets[0]))
    print("✅ Found Data Directory:", DATA_DIR)
else:
    DATA_DIR = "/kaggle/input/lifemate-zaic-2026-data/train+public_test"
    print("⚠️ Fallback Data Directory:", DATA_DIR)

os.environ["DATA_DIR"] = DATA_DIR

In [ ]:
# 5. Step 1: Generate Adaptive SFT Training Data from train.jsonl (with <thought> Meta-Reasoning & Distractor Invariance)
!python -u scripts/build_adaptive_sft_data.py \
    --output_dir data/adaptive_sft \
    --val_ratio 0.1 \
    --seed 42

In [ ]:
# 6. Step 2: Fine-Tune Qwen2.5-3B-Instruct via LoRA and Merge
import subprocess, sys, os, glob, zipfile

MODEL_DIR = "/kaggle/working/models/lifemate-adaptive-qwen"
LORA_DIR = "/kaggle/working/models/lifemate-adaptive-lora"

# Check if model was already merged in working dir
if os.path.exists(MODEL_DIR) and os.listdir(MODEL_DIR):
    print(f"✅ Found existing merged model in working dir: {MODEL_DIR}")
    print("⏩ Skipping training completely!")
    SKIP_TRAIN = True
else:
    SKIP_TRAIN = False

os.environ["MODEL_DIR"] = MODEL_DIR

if not SKIP_TRAIN:
    print("🚀 Starting Adaptive SFT Training on Qwen2.5-3B-Instruct (3 epochs, ~1.5h on GPU)...\n")
    cmd = [
        sys.executable, "-u", "train_adaptive_lora.py",
        "--model_name_or_path", "Qwen/Qwen2.5-3B-Instruct",
        "--train_file", "data/adaptive_sft/adaptive_train.jsonl",
        "--val_file", "data/adaptive_sft/adaptive_val.jsonl",
        "--output_dir", LORA_DIR,
        "--merged_dir", MODEL_DIR,
        "--epochs", "3",
        "--batch_size", "1",
        "--grad_accum", "8",
        "--lr", "1.5e-4",
        "--lora_r", "32",
        "--lora_alpha", "64",
        "--max_seq_length", "4096"
    ]
    print("Executing:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print(f"✅ Ready to run inference using model at {MODEL_DIR}")

In [ ]:
# 7. Step 3: Run Full Inference on public_test with Gipformer CPU ASR + Adaptive Agent
import os
sub_path = "/kaggle/working/submission.jsonl"
if os.path.exists(sub_path):
    with open(sub_path, "r", encoding="utf-8") as f_sub:
        count = sum(1 for line in f_sub if line.strip())
    if count < 634:
        print(f"Removing incomplete submission ({count}/634 lines) to regenerate cleanly...")
        os.remove(sub_path)
    else:
        print(f"Submission already complete ({count}/634 lines). Proceeding to validation...")

!rm -f /kaggle/working/transcripts_public_test.json
!python -u inference.py \
    --data_dir "$DATA_DIR" \
    --split public_test \
    --asr_engine gipformer \
    --asr_model g-group-ai-lab/gipformer1.5-68M-rnnt \
    --llm_model "$MODEL_DIR" \
    --output /kaggle/working/submission.jsonl

In [ ]:
# 8. Step 4: Clean, Deduplicate, and Validate Output with Official zaic_tools Validator
!python scripts/clean_submission.py --input /kaggle/working/submission.jsonl --data_dir "$DATA_DIR"

import os, subprocess
env = os.environ.copy()
env["PYTHONPATH"] = f"{DATA_DIR}/public_test:" + env.get("PYTHONPATH", "")
cmd = [
    "python", "-m", "zaic_tools",
    "--split", "public_test",
    "--root", DATA_DIR,
    "--validate", "/kaggle/working/submission.jsonl"
]
print("Running official zaic_tools validator...")
res = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print(res.stderr)

In [ ]:
# 9. Step 5: Check submission line count and preview
!wc -l /kaggle/working/submission.jsonl
!head -n 5 /kaggle/working/submission.jsonl

In [ ]:
# 10. Step 6: Backup LoRA Adapter for Future Runs (~115MB only)
import os, shutil
lora_dir = "/kaggle/working/models/lifemate-adaptive-lora"
if os.path.exists(lora_dir):
    shutil.make_archive("/kaggle/working/lifemate_adaptive_lora_adapter", "zip", lora_dir)
    print("LoRA adapter archived successfully at /kaggle/working/lifemate_adaptive_lora_adapter.zip")
    !ls -lh /kaggle/working/lifemate_adaptive_lora_adapter.zip
else:
    print("Adapter directory not found.")